# Score Images

## Setup

In [1]:
import os
import random
import sys
from pathlib import Path
from typing import Any, cast

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from PIL import Image
from tqdm.auto import tqdm

# Add the project root to sys.path to allow imports from avm package
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from lib.score import MODELS, DINOv2WithSealionScoringModel, OpenAIScoringModel
from lib.utils import load_info_records, detect_device

## Constants

In [2]:
INFO_TXT_PATH = "../data/bg1k_info.txt"
IMAGES_DIR = "bg1k_imgs"
FULL_IMAGES_DIR = f"../data/{IMAGES_DIR}/"
OUT_CSV_PATH = f"../data/out_scores/{IMAGES_DIR}.csv"

MAX_RECORDS: int | None = 20  # Set to None to score all records
SEED = 42

DINOV2_CHECKPOINT = PROJECT_ROOT / 'models/dinov2_vitb14_best.pt'  # Path to the DINOv2 finetuned checkpoint
BASE_URL = "https://api.sea-lion.ai/v1"
ENV_PATH = PROJECT_ROOT / ".env"
DEVICE = None  # None -> auto detect via detect_device()

OUTPUT_COLUMNS = [
    "image_name",
    "overall_score",
    "background_cleanliness",
    "text_watermark_score",
    "product_prominence",
    "commercial_appeal",
    "bg_class",
    "needs_bg_replacement",
    "reason",
]

## Load Client and Scoring Model

In [3]:
load_dotenv(ENV_PATH)

def load_client() -> OpenAI:
    api_key = os.getenv("SEALION_API_KEY")
    if not api_key:
        raise ValueError(f"SEALION_API_KEY not found. Ensure it is set in {ENV_PATH}.")
    client = OpenAI(api_key=api_key, base_url=BASE_URL)
    print(f"SEA-LION client ready.")
    return client

device = DEVICE or detect_device()
client = load_client()
sealion_model = OpenAIScoringModel(client=client, model_id=MODELS["sealionv4"])
scoring_model = DINOv2WithSealionScoringModel(
    sealion_model=sealion_model,
    checkpoint_path=str(DINOV2_CHECKPOINT),
    device=device,
)
print(f"Scoring model ready: {MODELS['dinov2-finetuned-with-sealionv4']}")

SEA-LION client ready.


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Scoring model ready: dinov2-finetuned-with-sealionv4


## Load Records

In [4]:
records = list(load_info_records(INFO_TXT_PATH))
if MAX_RECORDS is not None:
    records = random.Random(SEED).sample(records, min(MAX_RECORDS, len(records)))

print(f"Loaded {len(records)} records.")
for i, record in enumerate(records[:3], start=1):
    print(f"Example {i}: {record}")

Loaded 20 records.
Example 1: {'image_id': '654', 'image_name': '654.png', 'category': 'Warm milk sterilization', 'mask_image_name': '654_mask.png'}
Example 2: {'image_id': '114', 'image_name': '114.png', 'category': 'keyboard', 'mask_image_name': '114_mask.png'}
Example 3: {'image_id': '25', 'image_name': '25.png', 'category': 'fabric sofa', 'mask_image_name': '25_mask.png'}


## Resume and Prepare Pending Images

In [5]:
out_csv = Path(OUT_CSV_PATH)
out_csv.parent.mkdir(parents=True, exist_ok=True)

def _load_existing_results(out_csv: Path) -> tuple[list[dict[str, Any]], set[str]]:
    if not out_csv.exists() or out_csv.stat().st_size == 0:
        return [], set()

    existing = pd.read_csv(out_csv).reindex(columns=OUTPUT_COLUMNS)
    done: set[str] = set(existing["image_name"].astype(str)) if "image_name" in existing.columns else set()
    results = cast(list[dict[str, Any]], existing.to_dict(orient="records"))
    return results, done

results, done = _load_existing_results(out_csv)
if done:
    print(f"Resuming: {len(done)} images already scored.")
else:
    print("No existing CSV found. Starting fresh.")

pending_records = [record for record in records if record["image_name"] not in done]
print(f"Total selected: {len(records)} | Already scored: {len(done)} | To score: {len(pending_records)}")

No existing CSV found. Starting fresh.
Total selected: 20 | Already scored: 0 | To score: 20


## Score Images

In [6]:
def _error_score(reason: str) -> dict[str, Any]:
    return {
        "overall_score": -1,
        "background_cleanliness": -1,
        "text_watermark_score": -1,
        "product_prominence": -1,
        "commercial_appeal": -1,
        "bg_class": "",
        "needs_bg_replacement": None,
        "reason": reason,
    }

def _to_output_row(image_name: str, scored: dict[str, Any]) -> dict[str, Any]:
    return {
        "image_name": image_name,
        "overall_score": scored.get("overall_score", -1),
        "background_cleanliness": scored.get("background_cleanliness", -1),
        "text_watermark_score": scored.get("text_watermark_score", -1),
        "product_prominence": scored.get("product_prominence", -1),
        "commercial_appeal": scored.get("commercial_appeal", -1),
        "bg_class": scored.get("bg_class", ""),
        "needs_bg_replacement": scored.get("needs_bg_replacement", None),
        "reason": scored.get("reason", ""),
    }

def _save_results(results: list[dict[str, Any]], out_csv: Path) -> None:
    pd.DataFrame(results).reindex(columns=OUTPUT_COLUMNS).to_csv(out_csv, index=False)

In [7]:
success = 0
failed = 0
skipped = len(done)

try:
    for idx, record in enumerate(tqdm(pending_records, desc="Scoring"), start=1):
        image_name = record["image_name"]
        image_path = Path(FULL_IMAGES_DIR) / image_name

        if not image_path.exists():
            failed += 1
            row = _to_output_row(image_name, _error_score(f"missing_image: {image_path}"))
            results.append(row)
            print(f"[{idx}/{len(pending_records)}] Missing image: {image_path}")
            _save_results(results, out_csv)
            continue

        with Image.open(image_path) as img:
            scored = cast(dict[str, Any], scoring_model.score_image(img.convert("RGB")))

        row = _to_output_row(image_name, scored)
        results.append(row)

        if row["overall_score"] == -1:
            failed += 1
            print(f"[{idx}/{len(pending_records)}] Failed scoring: {image_name} | {row['reason']}")
        else:
            success += 1
            print(f"[{idx}/{len(pending_records)}] Scored: {image_name} -> {row['overall_score']}")

        _save_results(results, out_csv)
finally:
    print(f"Finished scoring: {success} succeeded, {failed} failed, {skipped} skipped.")
    print(f"CSV saved to: {out_csv}")

Scoring:   0%|          | 0/20 [00:00<?, ?it/s]

[1/20] Scored: 654.png -> 6.77
[2/20] Scored: 114.png -> 8.83
[3/20] Scored: 25.png -> 8.37
[4/20] Scored: 759.png -> 8.56
[5/20] Scored: 281.png -> 7.33
[6/20] Scored: 250.png -> 7.74
[7/20] Scored: 228.png -> 6.98
[8/20] Scored: 142.png -> 7.92
[9/20] Scored: 754.png -> 7.92
[10/20] Scored: 104.png -> 8.86
[11/20] Scored: 692.png -> 8.11
[12/20] Scored: 758.png -> 8.11
[13/20] Scored: 913.png -> 8.48
[14/20] Scored: 558.png -> 7.8
[15/20] Scored: 89.png -> 8.03
[16/20] Scored: 604.png -> 8.33
[17/20] Scored: 432.png -> 8.1
[18/20] Scored: 32.png -> 7.64
[19/20] Scored: 30.png -> 7.29
[20/20] Scored: 95.png -> 8.43
Finished scoring: 20 succeeded, 0 failed, 0 skipped.
CSV saved to: ../data/out_scores/bg1k_imgs.csv
